# Step 9 — I tre ranker, esplorati

`docs/09_ranker.md` spiega **perché** i ranker hanno questa forma. Questo
notebook mostra i numeri da cui quelle scelte discendono, nell'ordine in cui
sono stati trovati: prima la linea di base che rende il compito difficile,
poi il tetto che limita il ranker simbolico, poi il confronto.

Il riferimento non è annotato: è **la terapia che i cardiologi hanno
realmente prescritto alla dimissione**, già nei dati.


In [ ]:
import sys
from collections import Counter
from pathlib import Path

RADICE = Path.cwd().parent
sys.path.insert(0, str(RADICE / 'src'))

from ranker import (Caso, RankerContinuita, RankerFrequenza, RankerSimbolico,
                    RankerIbrido, INDICAZIONI, applica_filtro, carica_casi,
                    dividi, insieme_candidato, nomi_atc, nomi_icd)
from valuta_ranker import valuta_ranker, tetto_dei_candidati

casi = carica_casi(RADICE / 'data' / 'processed' / 'pipeline_b_v3')
addestramento, prova = dividi(casi, 0.3)
len(casi), len(addestramento), len(prova)

## 1. La linea di base che rende il compito difficile

Prima di costruire qualunque ranker: quanto della terapia di dimissione si
ottiene **copiando la terapia d'ingresso**? Se la risposta è «quasi tutta»,
un ranker misurato sulla terapia completa sta misurando il nulla.


In [ ]:
tot = sum(len(c.dimissione) for c in casi)
cont = sum(len(c.dimissione & c.terapia_ingresso) for c in casi)
sosp = sum(len(c.terapia_ingresso - c.dimissione) for c in casi)

print(f'classi alla dimissione, per ricovero : {tot/len(casi):.2f}')
print(f'  continuate dall\'ingresso          : {cont/tot:.1%}  ({cont})')
print(f'  nuove, decise durante il ricovero  : {(tot-cont)/tot:.1%}  ({tot-cont})')
print(f'  sospese durante il ricovero        : {sosp}')

Il 63,6% si ottiene senza ragionare. È per questo che lo step 9 misura **due
compiti** e considera vero solo il secondo: prevedere le classi **aggiunte**.


## 2. Il tetto del ranker simbolico, noto prima di scrivere una regola

Le linee guida che so citare sono cardiologiche. Quanta parte di ciò che un
cardiologo prescrive alla dimissione *è* cardiologia?


In [ ]:
def cardiovascolare(cls):
    return cls.startswith('C') or cls.startswith('B01')

tutte = Counter(c for caso in casi for c in caso.dimissione)
nuove = Counter(c for caso in casi for c in caso.aggiunte)
T, N = sum(tutte.values()), sum(nuove.values())

fuori = sum(v for c, v in tutte.items() if not cardiovascolare(c))
fuori_nuove = sum(v for c, v in nuove.items() if not cardiovascolare(c))
print(f'prescrizioni NON cardiovascolari      : {fuori/T:.1%}')
print(f'  fra le sole aggiunte                : {fuori_nuove/N:.1%}')

na = nomi_atc()
print('\nle otto classi non cardiovascolari più prescritte:')
for c, v in [(c, v) for c, v in tutte.most_common() if not cardiovascolare(c)][:8]:
    print(f'  {c}  {v:4}  (nuove {nuove[c]:3})  {na.get(c, "")}')

Il 40% del bersaglio è fuori dal perimetro delle linee guida cardiologiche.
**Non l'ho colmato inventando regole**: sarebbe stata conoscenza di un modello
travestita da linea guida, e il progetto vieta i mapping non citabili. Il
divario resta, ed è ciò che il ranker ibrido deve chiudere imparandolo.

È la stessa forma del risultato dello step 6: il richiamo della pipeline A era
limitato dal **denominatore della sua base di conoscenza**, non dal suo
algoritmo.


## 3. L'insieme candidato, e il tetto che impone

Costruito **solo** sui casi di addestramento. Prenderlo dal corpus intero
farebbe entrare le classi presenti solo nella prova, e il richiamo misurato
sarebbe gonfiato da una fuga di informazione.


In [ ]:
candidati = insieme_candidato(addestramento)
tetto = tetto_dei_candidati(prova, candidati)
print(f'classi candidate: {tetto["classi_candidate"]}')
print(f'tetto sulla terapia completa: {tetto["tetto_terapia_completa"]:.1%}')
print(f'tetto sulle aggiunte        : {tetto["tetto_aggiunte"]:.1%}')

# Quanto il filtro dello step 8 restringe davvero i candidati, paziente per
# paziente: è il punto in cui i due step si incastrano.
tolti = [len(candidati) - len(applica_filtro(c, candidati)) for c in prova]
print(f'\nclassi tolte dal filtro, in media: {sum(tolti)/len(tolti):.2f}'
      f'  (massimo {max(tolti)})')

## 4. Il confronto


In [ ]:
rankers = [RankerContinuita(), RankerFrequenza(), RankerSimbolico(), RankerIbrido()]
for r in rankers:
    r.addestra(addestramento)
risultati = [valuta_ranker(r, prova, candidati) for r in rankers]

for compito in ('terapia_completa', 'aggiunte'):
    print(f'\n=== {compito} ===')
    print(f'{"ranker":34} {"ric@3":>7} {"ric@5":>7} {"ric@10":>7} {"MAP":>7}')
    for r in risultati:
        m = r[compito]
        print(f'{r["nome"][:34]:34} {m["richiamo@3"]:6.1%} {m["richiamo@5"]:6.1%} '
              f'{m["richiamo@10"]:6.1%} {m["MAP"]:7.3f}')

Tre letture:

1. **Il compito 1 è vinto dalla linea di base banale**, e questo *è* il
   risultato del compito 1.
2. **Il simbolico da solo perde contro un ranker che non guarda il paziente.**
   Le linee guida cardiologiche, da sole, sono un ranker peggiore del sapere
   quali farmaci si prescrivono in questo reparto.
3. **L'ibrido supera la frequenza su tutte le metriche del compito 2**, con
   margini piccoli ma coerenti. **Rivisto allo step 11:** con un bootstrap su
   1 000 ricampionamenti dei ricoveri la differenza sul richiamo@5 sta in
   [−0,2%, +7,5%] e include lo zero. La direzione resta, il campione non basta
   a concluderlo. Vedi `docs/11_valutazione.md` sez. 6.


## 5. Un errore che ha dovuto essere corretto: guadagno contro probabilità

La prima versione dell'ibrido ordinava per **informazione mutua puntuale** e
andava peggio della frequenza. La PMI è un *guadagno*: una classe aggiunta a
metà dei pazienti qualunque sia la loro malattia ha PMI vicina a zero.
Ordinare per PMI mette in cima le classi **specifiche** e in fondo quelle
**probabili**, mentre la domanda del compito è quale classe verrà aggiunta.


In [ ]:
ibr = RankerIbrido(peso_guida=0.0)
ibr.addestra(addestramento)
esempio = prova[0]

# Per la stessa classe, il guadagno e la probabilità stimata ordinano diverso.
car = RankerIbrido._caratteristiche(esempio)
righe = []
for cls in candidati:
    g = max((ibr.pmi[(c, cls)] for c in car if (c, cls) in ibr.pmi), default=0.0)
    righe.append((cls, g, ibr._statistica(esempio, cls)))

print('prime 5 per GUADAGNO (la versione sbagliata):')
for cls, g, s in sorted(righe, key=lambda x: -x[1])[:5]:
    print(f'  {cls}  guadagno {g:5.2f}  log-prob {s:6.2f}  {na.get(cls, "")[:38]}')
print('\nprime 5 per PROBABILITÀ stimata (la versione corretta):')
for cls, g, s in sorted(righe, key=lambda x: -x[2])[:5]:
    print(f'  {cls}  guadagno {g:5.2f}  log-prob {s:6.2f}  {na.get(cls, "")[:38]}')

## 6. Il peso delle linee guida, tarato dove si deve

Su una parte di **validazione ritagliata dall'addestramento**, mai sulla
prova. Tarare sulla prova sarebbe scegliere il parametro guardando il
risultato che poi si riporta.


In [ ]:
sub, validazione = dividi(addestramento, 0.25, seme=777)
cand_v = insieme_candidato(sub)
print(f'{"peso":>6} {"ric@3":>7} {"ric@5":>7} {"ric@10":>7} {"MAP":>7}')
for peso in (0.0, 0.25, 0.5, 1.0, 2.0, 4.0, 8.0):
    r = RankerIbrido(peso_guida=peso)
    r.addestra(sub)
    m = valuta_ranker(r, validazione, cand_v)['aggiunte']
    print(f'{peso:6.2f} {m["richiamo@3"]:6.1%} {m["richiamo@5"]:6.1%} '
          f'{m["richiamo@10"]:6.1%} {m["MAP"]:7.3f}')

La curva è piatta fra 0,25 e 1,0 — dentro il pavimento di rumore del progetto
— e cala nettamente sopra: a peso alto la linea guida sovrasta il dato e il
ranker smette di sapere che in questo reparto si prescrivono gastroprotettori.

Il numero che conta è il primo: **a peso zero, sola statistica, il richiamo@3
è 28,3% contro il 31,2% dell'ottimo.** Le linee guida aggiungono qualcosa, ma
poco. È un risultato, non un difetto della taratura.


## 7. Il simbolico perde anche quando ha ragione

Un ricovero della prova: fibrillazione atriale parossistica, aterosclerosi,
già in terapia con anticoagulante diretto, antiaritmico e betabloccante.


In [ ]:
ni = nomi_icd()
c = next(x for x in prova if x.enc_oid == 10083543)
amm = applica_filtro(c, candidati)

print('condizioni:', '; '.join(f'{k} {ni.get(k, ni.get(k[:3], ""))}'
                              for k in sorted(c.condizioni)))
print('in terapia:', '; '.join(f'{k} {na.get(k, "")}' for k in sorted(c.terapia_ingresso)))
print('AGGIUNTE DAL MEDICO:', '; '.join(f'{k} {na.get(k, "")}'
                                      for k in sorted(c.aggiunte & set(amm))))

for nome, r in (('SIMBOLICO', rankers[2]), ('IBRIDO', rankers[3])):
    nuovi = [x for x in r.ordina(c, amm) if x.classe_atc not in c.terapia_ingresso]
    print(f'\n--- {nome} ---')
    for i, x in enumerate(nuovi[:5], 1):
        segno = '*' if x.classe_atc in c.aggiunte else ' '
        print(f' {segno}{i}. {x.classe_atc}  {na.get(x.classe_atc, "")[:44]}')
    print('    posizioni delle aggiunte reali:',
          [i for i, x in enumerate(nuovi, 1) if x.classe_atc in c.aggiunte])

Il simbolico propone `C10AA` **statine**; il medico ha prescritto `C10BA`
**statine in associazione**. È la stessa decisione clinica, e la metrica la
conta come errore. Quanto costa questo artefatto:


In [ ]:
def richiamo(r, profondita, k=5):
    """Richiamo@k contando l'azzeccato a profondità ATC data (micro-media)."""
    num = den = 0
    for caso in prova:
        ammessi = applica_filtro(caso, candidati)
        bersaglio = caso.aggiunte & set(ammessi)
        if not bersaglio:
            continue
        primi = [x.classe_atc for x in r.ordina(caso, ammessi)
                 if x.classe_atc not in caso.terapia_ingresso][:k]
        visti = {x[:profondita] for x in primi}
        num += len({b for b in bersaglio if b[:profondita] in visti})
        den += len(bersaglio)
    return num / den

print(f'{"ranker":12} {"ATC4":>8} {"ATC3":>8} {"ATC2":>8}')
for nome, r in (('simbolico', rankers[2]), ('frequenza', rankers[1]), ('ibrido', rankers[3])):
    print(f'{nome:12}' + ''.join(f'{richiamo(r, p):8.1%}' for p in (5, 4, 3)))

Allargare di un solo livello vale da 4 a 7 punti. È la motivazione diretta
della **metrica gerarchica dello step 11**: due terapie della stessa famiglia
non sono un errore quanto due terapie di famiglie diverse.


## 8. Le indicazioni e le loro fonti

Il vincolo di provenienza del progetto vale anche qui: nessuna regola senza
un documento pubblicato dietro.


In [ ]:
print(f'indicazioni: {len(INDICAZIONI)}')
print(f'  senza fonte: {sum(1 for i in INDICAZIONI if not i.fonte.strip())}')
print(f'  con un fatto che il sistema non estrae: '
      f'{sum(1 for i in INDICAZIONI if i.fatto_non_estratto)}')
print(f'  innescate dalla terapia e non da una diagnosi: '
      f'{sum(1 for i in INDICAZIONI if i.atc_richiesto)}')

print('\ndistribuzione per classe di raccomandazione ESC:')
for k, v in sorted(Counter(i.classe_racc for i in INDICAZIONI).items()):
    print(f'  classe {k:3}  {v}')

print('\ni fatti che le linee guida userebbero e il sistema non ha:')
for f in sorted({i.fatto_non_estratto for i in INDICAZIONI if i.fatto_non_estratto}):
    print(f'  - {f}')